# M2.1: Market clearing — order books, supply and demand, welfare

## Objectives

At the end of this notebook you should be able to:

- read a day-ahead **order book** of sell offers and buy bids, and build the aggregate **supply and demand curves**
- clear a **uniform-price (pay-as-clear) auction** by hand and in code, including which order sets the price
- write market clearing as a **welfare-maximisation LP** and recognise the clearing price as its dual (Maths-2)
- compute **consumer and producer surplus**, and explain why producer surplus (inframarginal rent) is what pays for power plants
- clear the same market in **PyPSA** — and avoid a sign trap with demand bids

In M1.2 you dispatched generators against a fixed load and read the price as a dual. A real power exchange works from the other side: participants submit **orders**, and the exchange finds the price at which supply meets demand. Both views give the same answer, and this lesson shows why. The order book is **synthetic** but sized like a Croatian evening hour.

In [1]:
import logging
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pypsa
from scipy.optimize import linprog

logging.getLogger("pypsa").setLevel(logging.ERROR)
logging.getLogger("linopy").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=FutureWarning)

### The order book

In the European day-ahead market every participant submits, for each hour (or quarter-hour) of the next day, orders of the form *"sell up to q MW at no less than p €/MWh"* or *"buy up to q MW at no more than p €/MWh"*. The coupled European exchanges clear all of them together. Here is one hour of a synthetic order book:

In [2]:
offers = pd.DataFrame({
    "order": ["solar", "wind", "run_of_river", "nuclear_share", "coal", "ccgt", "imports", "ocgt"],
    "price": [-5.0, 0.0, 5.0, 12.0, 95.0, 110.0, 120.0, 165.0],      # EUR/MWh, minimum price to sell
    "qty":   [300.0, 400.0, 500.0, 350.0, 210.0, 500.0, 600.0, 250.0],  # MW
})
bids = pd.DataFrame({
    "order": ["inflexible_load", "industry_A", "industry_B", "pumped_storage", "electrolyser"],
    "price": [3000.0, 140.0, 115.0, 60.0, 40.0],                      # EUR/MWh, maximum price to buy
    "qty":   [1700.0, 200.0, 150.0, 120.0, 100.0],                    # MW
})
print(f"offered {offers.qty.sum():,.0f} MW, bid {bids.qty.sum():,.0f} MW")
offers

offered 3,110 MW, bid 2,270 MW


           order  price    qty
0          solar   -5.0  300.0
1           wind    0.0  400.0
2   run_of_river    5.0  500.0
3  nuclear_share   12.0  350.0
4           coal   95.0  210.0
5           ccgt  110.0  500.0
6        imports  120.0  600.0
7           ocgt  165.0  250.0

Three things to notice:

- **Most demand is inflexible**: households and most businesses do not react to the hourly price, so suppliers bid that volume at a very high price (here an illustrative 3,000 €/MWh) — "buy whatever it costs".
- **Some demand is price-sensitive**: industry, pumped storage (pumping only when power is cheap) and an electrolyser.
- **Solar offers at −5 €/MWh**: a plant that receives a subsidy per MWh produced can still earn money at a slightly negative price, so it offers below zero. Negative prices are the subject of M2.2.

### Supply and demand curves

Sort the offers from cheapest to most expensive and add up their quantities: the **supply curve** (the merit order of M1.2). Sort the bids from highest to lowest price: the **demand curve**. Where they cross, the market clears.

In [3]:
def step_curve(orders, ascending):
    """Cumulative quantity (MW) and price (EUR/MWh) of a step curve, ready for plotting."""
    o = orders.sort_values("price", ascending=ascending)
    q = np.r_[0, o.qty.cumsum().to_numpy()]
    return q, np.r_[o.price.to_numpy(), o.price.to_numpy()[-1]]


fig, ax = plt.subplots(figsize=(9, 4.5))
qs, ps = step_curve(offers, ascending=True)
qd, pd_ = step_curve(bids, ascending=False)
ax.step(qs, ps, where="post", label="supply (offers)", lw=2)
ax.step(qd, pd_, where="post", label="demand (bids)", lw=2)
ax.plot(2050, 110, "ko", ms=7, label="clearing point: 2,050 MW at 110 EUR/MWh")
ax.set(ylim=(-20, 200), xlabel="quantity (MW)", ylabel="price (EUR/MWh)",
       title="Aggregate curves for one hour (demand above 200 EUR/MWh cut off)")
ax.legend()
plt.show()

### Clearing by walking the curves

Pair the cheapest remaining offer with the highest remaining bid, trade as much as both allow, and repeat — as long as the buyer is willing to pay at least what the seller asks:

In [4]:
def clear_market(offers, bids):
    """Uniform-price clearing by walking the curves. Returns volume, price and accepted quantities."""
    o = offers.sort_values("price").reset_index(drop=True)
    b = bids.sort_values("price", ascending=False).reset_index(drop=True)
    left_o, left_b = o.qty.to_numpy().copy(), b.qty.to_numpy().copy()
    i = j = 0
    while i < len(o) and j < len(b) and o.price[i] <= b.price[j]:
        q = min(left_o[i], left_b[j])
        left_o[i] -= q
        left_b[j] -= q
        last_offer, last_bid = i, j
        if left_o[i] == 0:
            i += 1
        if left_b[j] == 0:
            j += 1
    sold, bought = o.qty - left_o, b.qty - left_b
    if 0 < sold[last_offer] < o.qty[last_offer]:
        price = o.price[last_offer]                 # a partly accepted offer sets the price
    elif 0 < bought[last_bid] < b.qty[last_bid]:
        price = b.price[last_bid]                   # a partly accepted bid sets the price
    else:
        price = None                                # no partial order: a price range (see below)
    return {"volume": sold.sum(), "price": price,
            "sold": dict(zip(o.order, sold)), "bought": dict(zip(b.order, bought))}


result = clear_market(offers, bids)
print(f"cleared {result['volume']:,.0f} MW at {result['price']:.2f} EUR/MWh")
print("accepted offers:", {k: v for k, v in result["sold"].items() if v > 0})
print("accepted bids:  ", {k: v for k, v in result["bought"].items() if v > 0})

cleared 2,050 MW at 110.00 EUR/MWh
accepted offers: {'solar': 300.0, 'wind': 400.0, 'run_of_river': 500.0, 'nuclear_share': 350.0, 'coal': 210.0, 'ccgt': 290.0}
accepted bids:   {'inflexible_load': 1700.0, 'industry_A': 200.0, 'industry_B': 150.0}


The market clears **2,050 MW at 110 €/MWh**. The CCGT is the **marginal** order: it sells only 290 of its 500 MW, so it is exactly indifferent at 110 €/MWh and sets the price. Every accepted seller receives 110 €/MWh, every accepted buyer pays 110 €/MWh — that is the **uniform price** or *pay-as-clear* rule. Pumped storage (bid 60) and the electrolyser (bid 40) are not served: power is too expensive for them in this hour. Imports and the OCGT are not needed.

When **no** order is partially accepted, the curves cross on a vertical step and every price in a range clears the same volume. Then a market rule must pick one price (the exercise explores this case).

### Clearing as an optimisation: welfare maximisation

A power exchange does not literally walk the curves — with many hours, zones and complex orders it solves an optimisation problem. For one zone and one hour it is an LP that **maximises welfare**, the value of everything traded to buyers minus its cost to sellers:

**maximise** Σ_bids p_b·d_b − Σ_offers p_o·s_o
**subject to** Σ s_o = Σ d_b (balance), 0 ≤ s_o ≤ q_o, 0 ≤ d_b ≤ q_b

By Maths-2 the **dual of the balance constraint is the clearing price**:

In [5]:
n_o = len(offers)
cost = np.r_[offers.price, -bids.price]                       # linprog minimises: -welfare
balance = [np.r_[np.ones(n_o), -np.ones(len(bids))]]
lp = linprog(cost, A_eq=balance, b_eq=[0.0], bounds=list(zip(np.zeros(len(cost)), np.r_[offers.qty, bids.qty])),
             method="highs")
lp_sold, lp_bought = lp.x[:n_o], lp.x[n_o:]
lp_price, welfare = lp.eqlin.marginals[0], -lp.fun
assert np.isclose(lp_sold.sum(), result["volume"]) and np.isclose(lp_price, result["price"])
print(f"LP: {lp_sold.sum():,.0f} MW at {lp_price:.2f} EUR/MWh, welfare {welfare:,.0f} EUR/h")

LP: 2,050 MW at 110.00 EUR/MWh, welfare 5,088,200 EUR/h


Same volume, same price. The equivalence is no coincidence: the KKT conditions of this LP (Maths-3) say that every accepted offer has p_o ≤ price, every accepted bid has p_b ≥ price, and partly accepted orders sit exactly at the price — which is the walking rule.

### Who gains? Consumer and producer surplus

Welfare splits between the two sides of the market:

- **consumer surplus** = Σ (bid price − clearing price) × quantity bought — what buyers were willing to pay above what they paid
- **producer surplus** = Σ (clearing price − offer price) × quantity sold — what sellers earn above their asking price

In [6]:
sold = pd.Series(lp_sold, index=offers.order)
bought = pd.Series(lp_bought, index=bids.order)
producer = ((lp_price - offers.price.to_numpy()) * lp_sold)
consumer = ((bids.price.to_numpy() - lp_price) * lp_bought)
assert np.isclose(producer.sum() + consumer.sum(), welfare), "surpluses add up to welfare"
pd.DataFrame({"sold_MW": sold, "producer_surplus_EUR_h": producer}, index=offers.order).round(0) + 0.0   # + 0.0 turns -0.0 into 0.0

               sold_MW  producer_surplus_EUR_h
order                                         
solar            300.0                 34500.0
wind             400.0                 44000.0
run_of_river     500.0                 52500.0
nuclear_share    350.0                 34300.0
coal             210.0                  3150.0
ccgt             290.0                     0.0
imports            0.0                     0.0
ocgt               0.0                     0.0

Producer and consumer surplus add up exactly to the LP's welfare. The absolute consumer surplus is dominated by the inflexible load's 3,000 €/MWh bid and has little meaning on its own — changes in it do (M2.2).

Look at the producer surplus: run-of-river earns 105 €/MWh above its offer, nuclear 98 €/MWh, while the price-setting CCGT earns nothing. This **inframarginal rent** is not a windfall to be removed; it is how plants with low running costs but high construction costs (hydro dams, nuclear, wind) recover their investment in an energy-only market. Whether these rents are large enough to pay for new plants is the *missing money* debate (Module 05).

### The same market in PyPSA — and a sign trap

PyPSA has no "bid" component, but a demand bid is a generator with **sign = −1** (it withdraws power) whose marginal cost is the **negative** of the bid price — buying at p_b creates value p_b, which a cost-minimiser sees as a cost of −p_b:

In [7]:
def pypsa_market(bid_cost_sign):
    n = pypsa.Network()
    n.set_snapshots([0])
    n.add("Bus", "HR")
    for r in offers.itertuples():
        n.add("Generator", r.order, bus="HR", p_nom=r.qty, marginal_cost=r.price)
    for r in bids.itertuples():
        n.add("Generator", r.order, bus="HR", p_nom=r.qty, sign=-1, marginal_cost=bid_cost_sign * r.price)
    n.optimize(solver_name="highs", log_to_console=False)
    return n


n = pypsa_market(bid_cost_sign=-1)
price = n.buses_t.marginal_price.iloc[0, 0]
assert np.isclose(price, lp_price) and np.isclose(-n.objective, welfare)
print(f"PyPSA (marginal_cost = -bid): price {price:.2f} EUR/MWh, welfare {-n.objective:,.0f} EUR/h")

wrong = pypsa_market(bid_cost_sign=+1)
print(f"PyPSA (marginal_cost = +bid): price {wrong.buses_t.marginal_price.iloc[0, 0]:.2f} EUR/MWh, "
      f"bids served {wrong.generators_t.p.iloc[0][bids.order].sum():.0f} MW")

PyPSA (marginal_cost = -bid): price 110.00 EUR/MWh, welfare 5,088,200 EUR/h


PyPSA (marginal_cost = +bid): price -5.00 EUR/MWh, bids served 0 MW


With the correct sign PyPSA reproduces the price and the welfare exactly. With the intuitive but wrong sign, consuming power looks like a *cost*, so the optimiser serves **no demand at all**, reports a price of −5 €/MWh — and raises no error. Whenever you model flexible demand, check that it is actually served when the price is below its bid.

## Check your understanding

**Clearing**

1. The inflexible load rises from 1,700 to 2,000 MW. Predict the new marginal order and price before computing them.
2. Why does every accepted seller receive the same price, even solar, which offered at −5 €/MWh? What would change under *pay-as-bid*, and why would bidders then change their offers?

**Welfare and surplus**

1. Which orders' surplus is zero in this hour, and why?
2. The electrolyser raises its bid from 40 to 120 €/MWh. Does welfare rise or fall? Does the price change?